In [1]:
import pandas

bbx= 37.708435791447656, -122.51188063090561 37.840348910058665, -122.3592207270698

to prepare the data for the analysis, we need to filter the data to only include the san_francisco area of interest. The following SQL commands can be used to create new tables and export them to CSV files:

```sql
CREATE TABLE osm_san_francisco AS
SELECT * FROM osm
WHERE ST_Within(osm_geom, ST_MakeEnvelope
(-122.51188063090561,37.708435791447656, -122.3592207270698,37.840348910058665, 4326));
\copy osm_san_francisco TO 'osm_san_francisco.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```

```sql
CREATE TABLE foursquare_san_francisco AS
SELECT * FROM foursquare
WHERE ST_Within(fsq_geom, ST_MakeEnvelope
(-122.51188063090561,37.708435791447656, -122.3592207270698,37.840348910058665, 4326));
\copy foursquare_san_francisco TO 'foursquare_san_francisco.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```

```sql
CREATE TABLE fsq_osm_san_francisco AS
SELECT * FROM fsq_osm_filtered_5_lev 
WHERE ST_Within(fsq_geom, ST_MakeEnvelope
(-122.51188063090561,37.708435791447656, -122.3592207270698,37.840348910058665, 4326));
\copy fsq_osm_san_francisco TO 'fsq_osm_san_francisco.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```

In [2]:
from shapely.geometry import Point, Polygon
san_francisco_polygon = [(37.70799698895251, -122.50627681067586),(37.79214368938928, -122.51587899722284),(37.81439684874012, -122.48151769519464),(37.85642548785611, -122.3939741349364),(37.75935248585031, -122.34163101381763),(37.70886909298355, -122.34931785298726),(37.70799698895251, -122.50627681067586)]
def get_filtered_df(df, lat_col, lon_col, polygon):
    poly = Polygon([(lon, lat) for lat, lon in polygon])  # shapely uses (x=lon, y=lat)
    mask = df.apply(lambda row: poly.contains(Point(row[lon_col], row[lat_col])), axis=1)
    return df[mask]

def get_bounded_df(df, lat_col, lon_col, lat_min, lat_max, lon_min, lon_max):
    bounded_df = df[(df[lat_col] >= lat_min) & (df[lat_col] <= lat_max) & (df[lon_col] >= lon_min) & (df[lon_col] <= lon_max)]
    return bounded_df

def get_osm_processed_df(df):
    df['osm_name'] = df.apply(lambda row: get_name(row, 'osm_name'), axis=1)
    return df

def get_name(row, col_name):
    name = row[col_name]
    if pandas.isna(name):
        return name
    if '"name"=>"' in name:
        name = name.split('"name"=>"')[1].split('"')[0]
    elif '"name:en"=>"' in name:
        name = name.split('"name:en"=>"')[1].split('"')[0]
    elif '"name:fr"=>"' in name:
        name = name.split('"name:fr"=>"')[1].split('"')[0]
    elif '"name:da"=>"' in name:
        name = name.split('"name:da"=>"')[1].split('"')[0]
    elif '"name:de"=>"' in name:
        name = name.split('"name:de"=>"')[1].split('"')[0]
    elif '"name:"=>"' in name:
        name = name.split('"name:"=>"')[1].split('"')[0]
    elif 'name:' in name:
        print("Unexpected name format:", name)
        name = name
    else:
        name = None
    return name

def get_fsq_processed_df(df):   
    df['fsq_name'] = df['fsq_name'].apply(lambda x: x if (not pandas.isna(x) and len(x) > 10) else None)
    return df

In [3]:
osm_data = pandas.read_csv("../../data/osm_san_francisco.csv")
osm_filtered = get_filtered_df(osm_data, 'osm_latitude', 'osm_longitude', san_francisco_polygon)
osm_filtered = get_osm_processed_df(osm_filtered)


Unexpected name format: "name:ru"=>"на крыше института искусств"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "alt_name"=>"Greater Farralones National Marine Sacntuary Headquarters", "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "old_name:1915-1990"=>"Fort Point Coast Guard Station"
Unexpected name format: "alt_name"=>"Greater Farralones National Marine Sacntuary Headquarters", "old_name:1915-1990"=>"Fort Point Coast Guard Station"

In [4]:
fsq_data = pandas.read_csv("../../data/fsq_san_francisco.csv")
fsq_filtered = get_filtered_df(fsq_data, 'fsq_latitude', 'fsq_longitude', san_francisco_polygon)
# fsq_filtered = get_fsq_processed_df(fsq_filtered)

In [5]:
fsq_osm_data = pandas.read_csv("../../data/fsq_osm_san_francisco.csv", low_memory=False)
fsq_osm_data = fsq_osm_data[fsq_osm_data['fsq_osm_name_similarity_score_lev'] > 0.5]
fsq_osm_filtered = get_filtered_df(fsq_osm_data, 'fsq_latitude', 'fsq_longitude', san_francisco_polygon)

In [6]:
population_data = pandas.read_csv("../../data/population.csv")
population_data_filtered = get_filtered_df(population_data, 'Y', 'X', san_francisco_polygon)

In [44]:

def show_filtered_data(df,cols,limit=5,label="", shuffle=False, seed=42):
    if shuffle:
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    # cols = [cols[0]]
    df = df.dropna(subset=cols)
    print(f"\\begin{{table}}[h]")
    print(f"\\caption{{San Francisco Sample {label.replace('_', ' ')}}}")
    print(f"\\label{{tab:sanfrancisco_sample_{label}}}")
    print(f"\\begin{{tabular}}{{| {' | '.join([f'm{{0.2\\linewidth}}'] * (len(cols)+1))} |}}")
    print(f"\\hline")
    # df = df.sort_values(by=cols[0])
    print(' & '.join([f"\\textbf{{{col}}}" for col in cols]).replace('_', '\\_') + ' & \\textbf{Observation}\\\\ \\hline')
    counter = 0
    for row in df.itertuples():
        check = False
        for col in cols:
            if pandas.isna(getattr(row, col)):
                check = True
                break
            # if its not in english:
            if isinstance(getattr(row, col), str) and not all(ord(c) < 128 for c in getattr(row, col)):
                check = True
                break
            # if too long:
            if isinstance(getattr(row, col), str) and len(getattr(row, col)) > 44:
                check = True
                break
        if check:
            continue
        else:
            for col in cols[:-1]:
                print(f"{getattr(row, col)}".replace('_', '\\_').replace('&', '\\&'), end=" & ")
            print(f"{getattr(row, cols[-1])}".replace('_', '\\_').replace('&', '\\&'), end=" & ")
            print(" \\\\ \\hline")
        counter += 1
        if counter >= limit:
            break
    print(f"\\end{{tabular}}")
    print(f"\\end{{table}}")
l = 50
show_filtered_data(fsq_filtered, ['fsq_name', 'fsq_category_labels'], limit=l, label="fsq", shuffle=True)
show_filtered_data(osm_filtered, ['osm_name','osm_class','osm_type'], limit=l, label="osm", shuffle=True)
show_filtered_data(fsq_osm_filtered, ['fsq_name']+['osm_name','osm_type'], limit=l, label="fsq_osm", shuffle=True)



\begin{table}[h]
\caption{San Francisco Sample fsq}
\label{tab:sanfrancisco_sample_fsq}
\begin{tabular}{| m{0.2\linewidth} | m{0.2\linewidth} | m{0.2\linewidth} |}
\hline
\textbf{fsq\_name} & \textbf{fsq\_category\_labels} & \textbf{Observation}\\ \hline
Kaiser Blood Pressure Clinic & ['Health and Medicine > Medical Lab'] &  \\ \hline
Supreme Hair Cuts & ['Retail > Cosmetics Store'] &  \\ \hline
Voting at HAMPTON COURT & ['Community and Government > Polling Place'] &  \\ \hline
Holly Jolly Bar & ['Dining and Drinking > Bar'] &  \\ \hline
San Francisco Medical Society & ['Health and Medicine > Medical Center'] &  \\ \hline
Lola of San Francisco & ['Retail > Fashion Retail > Clothing Store'] &  \\ \hline
Landor San Francisco & ['Landmarks and Outdoors > Structure'] &  \\ \hline
Belle Cose & ['Retail > Antique Store'] &  \\ \hline
Millenium Construction Corp. & ['Retail > Construction Supplies Store'] &  \\ \hline
MadKat & ['Retail > Cosmetics Store'] &  \\ \hline
Abacus Row & ['Retail > 

In [8]:
cols = ['fsq_name', 'fsq_category_labels', 'fsq_latitude','fsq_longitude']
fsq_filtered[cols].head(100)


,fsq_name,fsq_category_labels,fsq_latitude,fsq_longitude
0,33rd,NaN,37.755315,-122.491136
1,Hat party,NaN,37.765397,-122.428548
2,Cream of Beat,['Arts and Entertainment > Night Club'],37.782593,-122.408055
3,Osumo,NaN,37.792760,-122.397685
4,Social Media Week: Klout Office,NaN,37.782144,-122.395880
...,...,...,...,...
96,Below Market,['Sports and Recreation > Gym and Studio'],37.789274,-122.400993
97,RadiumOne Minsk,['Business and Professional Services > Office ...,37.788846,-122.400179
98,85 2nd St Parking Garage,['Travel and Transportation > Parking'],37.788347,-122.399863
99,FDIC,['Landmarks and Outdoors > Structure'],37.789616,-122.398647


In [9]:
cols = ['osm_name','osm_class','osm_type', 'osm_latitude','osm_longitude']
# cols = osm_filtered.columns.tolist()
osm_filtered[cols].head(100)

,osm_name,osm_class,osm_type,osm_latitude,osm_longitude
0,Fat Beli Deli,amenity,pub,37.728887,-122.404354
1,Jasmine Tea House,amenity,restaurant,37.744799,-122.420234
2,Pikabootique,shop,toys,37.751103,-122.434215
3,DeStore,shop,yes,37.777101,-122.422123
4,PG&E,landuse,industrial,37.760875,-122.415374
...,...,...,...,...,...
95,Mother's Building,building,yes,37.734762,-122.503589
96,Playfield Lawn,landuse,grass,37.734593,-122.502920
97,Cassowary,natural,grassland,37.734766,-122.501809
98,Cassowary,tourism,attraction,37.734766,-122.501809


In [10]:
cols = ['fsq_name',  'osm_name','osm_type', 'fsq_latitude','fsq_longitude','osm_latitude','osm_longitude', 'fsq_category_labels']
cols = fsq_osm_filtered.columns.tolist()
data = fsq_osm_filtered[cols].head(100)
data
# print(','.join(cols))
# for row in data.itertuples():
#     print(f"{row.fsq_name}\t & {row.osm_name}\t & {row.osm_type}\t & \\\\ \hline\n{row.fsq_latitude},{row.fsq_longitude}\n{row.osm_latitude},{row.osm_longitude}\n{row.fsq_category_labels}\n")

,fsq_place_id,fsq_name,fsq_latitude,fsq_longitude,fsq_address,fsq_locality,fsq_region,fsq_postcode,fsq_admin_region,fsq_post_town,...,osm_name,osm_address,osm_extratags,osm_geometry,osm_latitude,osm_longitude,osm_geom,fsq_osm_name_similarity_score_trg,fsq_osm_distance,fsq_osm_name_similarity_score_lev
0,4f11da59e4b06784113de370,SF Philadelphian Seventh-day Adventist Church,37.786518,-122.439181,NaN,San Francisco,CA,94115,NaN,NaN,...,Philadelphian Seventh Day Adventist Church,NaN,"""ele""=>""46"", ""religion""=>""christian"", ""verifie...",0103000020E61000000100000029000000728E95F3209C...,37.786561,-122.439229,0101000020E610000033BD81521C9C5EC0529CD306AEE4...,0.952381,0.000064,0.888889
1,655e991cc72849099cf25ea9,Marina Greens,37.798157,-122.435901,NaN,San Francisco,CA,94123,NaN,NaN,...,Marina Greens,"""street""=>""Fillmore Street"", ""housenumber""=>""3...","""check_date""=>""2024-10-12""",0101000020E610000026074724E59B5EC097CF4DF62AE6...,37.798186,-122.435861,0101000020E610000026074724E59B5EC097CF4DF62AE6...,1.000000,0.000050,1.000000
2,4c78bdfebd346dcbe246f3ef,Monaghans Two,37.800209,-122.439664,NaN,San Francisco,CA,94123,NaN,NaN,...,Monaghan's,"""street""=>""Pierce Street"", ""housenumber""=>""324...","""height""=>""7""",0103000020E6100000010000000B000000F85C5C99269C...,37.800045,-122.439726,0101000020E6100000D72BCF76249C5EC0C5D0DBDB67E6...,0.470588,0.000176,0.615385
3,4f122595e4b07e9ecb39bc16,Wellsfargo,37.797740,-122.430707,NaN,San Francisco,CA,94123,NaN,NaN,...,Wells Fargo,NaN,"""operator""=>""Wells Fargo"", ""brand:wikidata""=>""...",0101000020E6100000A35A4414939B5EC09770E82D1EE6...,37.797796,-122.430852,0101000020E6100000A35A4414939B5EC09770E82D1EE6...,0.642857,0.000155,0.818182
4,4f122595e4b07e9ecb39bc16,Wellsfargo,37.797740,-122.430707,NaN,San Francisco,CA,94123,NaN,NaN,...,Wells Fargo,NaN,"""brand:wikidata""=>""Q744149""",0101000020E6100000CD806907929B5EC01492CCEA1DE6...,37.797788,-122.430788,0101000020E6100000CD806907929B5EC01492CCEA1DE6...,0.642857,0.000094,0.818182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,586a8674761b1a35b5945e7e,Fitness SF Embarcadero Center,37.795189,-122.398351,2 Embarcadero Ctr,San Francisco,CA,94111,NaN,NaN,...,Embarcadero Center 2,NaN,"""area""=>""yes""",0103000020E6100000010000000D000000A822707F8999...,37.794862,-122.398460,0101000020E6100000FADFA05E80995EC05813510ABEE5...,0.593750,0.000345,0.551724
96,586a8674761b1a35b5945e7e,Fitness SF Embarcadero Center,37.795189,-122.398351,2 Embarcadero Ctr,San Francisco,CA,94111,NaN,NaN,...,Embarcadero Center 2,NaN,"""height""=>""130"", ""wikidata""=>""Q7858934"", ""buil...",0103000020E610000001000000140000006792A2DF8899...,37.794997,-122.398495,0101000020E6100000EF8C6AF080995EC089A1AE73C2E5...,0.593750,0.000240,0.551724
97,5473ef58498e88accd5249dc,Kirimachi Ramen,37.795386,-122.397891,3 Embarcadero Ctr,San Francisco,CA,94111,NaN,NaN,...,Kirimachi Ramen,NaN,"""level""=>""0"", ""phone""=>""+1 415 8729171"", ""cuis...",0101000020E61000003D1E447C71995EC02261CE22CAE5...,37.795231,-122.397552,0101000020E61000003D1E447C71995EC02261CE22CAE5...,1.000000,0.000373,1.000000
98,4fbacf70e4b0b68bd9ee2c78,Two Embarcadero Center Parking,37.794762,-122.398088,2 Embarcadero Ctr,San Francisco,CA,94111,NaN,NaN,...,Embarcadero Center 2,NaN,"""area""=>""yes""",0103000020E6100000010000000D000000A822707F8999...,37.794862,-122.398460,0101000020E6100000FADFA05E80995EC05813510ABEE5...,0.575758,0.000385,0.633333


In [11]:
osm_filtered.to_csv("../../data/osm_san_francisco_filtered.csv", index=False)
fsq_filtered.to_csv("../../data/fsq_san_francisco_filtered.csv", index=False)
fsq_osm_filtered.to_csv("../../data/fsq_osm_san_francisco_filtered.csv", index=False)
population_data_filtered.to_csv("../../data/population_san_francisco_filtered.csv", index=False)